In [84]:
# to configure the llm to load

import os
import dotenv

dotenv.load_dotenv("../.env")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ")


# model = "openai/gpt-oss-120b"
# model = "openai/gpt-oss-20b"
model = "llama-3.3-70b-versatile"
# model = "meta-llama/llama-4-scout-17b-16e-instruct"
# model = "deepseek-r1-distill-llama-70b"

In [85]:
# to load the model from Groq using langchain
# init_chat_model()

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model=model,
    model_provider="groq",
    temperature=0
)

# to check the model
response = llm.invoke("What is the capital of France?")
print(response)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.007140474, 'completion_tokens_details': None, 'prompt_time': 0.006416761, 'prompt_tokens_details': None, 'queue_time': 0.187541174, 'total_time': 0.013557235}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_e65acd3773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2ee0-030f-74a0-82b8-4c1d8e3c0878-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


In [86]:
# tools for agents
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper

# a tool to search web
tool_search = DuckDuckGoSearchResults()

# a tool to query wikipedia
wiki_api = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=10000
)
tool_wiki = WikipediaQueryRun(api_wrapper=wiki_api)

tool_set = [tool_search, tool_wiki]


In [87]:
from pprint import pprint
llm_with_tools = llm.bind_tools(tool_set)

response = llm_with_tools.invoke("Who is the president of the United States?")
pprint(response)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dkmvspzx5', 'function': {'arguments': '{"query":"current president of the United States"}', 'name': 'duckduckgo_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 395, 'total_tokens': 418, 'completion_time': 0.04056346, 'completion_tokens_details': None, 'prompt_time': 0.137771962, 'prompt_tokens_details': None, 'queue_time': 0.202132686, 'total_time': 0.178335422}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2ee0-1c0e-7cc0-b721-d7b412c0e1ae-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'current president of the United States'}, 'id': 'dkmvspzx5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 395, 'output_tokens': 23, 'total_tokens': 418})


In [88]:
pprint(response.tool_calls)
# for tool in tool_set:
#     print(tool.name)

[{'args': {'query': 'current president of the United States'},
  'id': 'dkmvspzx5',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'}]


In [89]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
# to bind the tools to the llm
llm_with_tools = llm.bind_tools(tool_set)

# now need to define manual tool calling
tool_mapping = {
    "duckduckgo_results_json": tool_search,
    "wikipedia": tool_wiki
}

chat_history = []

# system prompt to guide the llm to use the tools effectively
RESEARCH_SYSTEM_PROMPT = """
You are an expert researcher. Use tools multiple times to verify facts.
Identify gaps in information and perform follow-up searches.
"""

chat_history.append(SystemMessage(content=RESEARCH_SYSTEM_PROMPT))  

# user_query = """
# why nvidia is not seeing the GPU development by AMD, apple and intel as a threat, however the 
# A15 and A16 chips from Tesla as a serious threat?
# """

user_query = """
why the statement by the Pakistan military chief Asif Munir, Jinna was a Shia, is controversial?
"""


chat_history.append(HumanMessage(content=user_query))

def call_tool(response):
    calls = response.tool_calls
    tool_messages = []

    for call in calls:
        # print(call.get("name"))
        tool_name = call.get("name")
        args = call.get("args")

        tool = tool_mapping.get(tool_name)
        if tool:
            observation = tool.invoke(args)
            tool_messages.append(
                ToolMessage(
                    content=str(observation), 
                    tool_call_id=call.get("id")
                )
            )

        else:
            tool_messages.append(
                ToolMessage(
                    content=f"Tool '{tool_name}' not found in tool mapping.", 
                    tool_call_id=response.call.get("id")
                )
            )  
    return tool_messages
    
while True:
    response = llm_with_tools.invoke(chat_history)
    # need to append the reponse to the chat_history to maintain the context for the llm
    chat_history.append(response)

    if response.tool_calls:
        tool_response = call_tool(response)
        pprint(response.tool_calls)
        chat_history.extend(
            # append the tool response with the id to link it to the tool call in the response
            tool_response
        )
    else:
        print(response.content)
        break 

[{'args': {'query': 'Asif Munir Jinnah Shia controversy'},
  'id': '2p9f1nxgt',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'},
 {'args': {'query': 'Asif Munir'},
  'id': '0tfyd50df',
  'name': 'wikipedia',
  'type': 'tool_call'},
 {'args': {'query': 'Muhammad Ali Jinnah'},
  'id': 'q6ec2xcgt',
  'name': 'wikipedia',
  'type': 'tool_call'},
 {'args': {'query': 'Jinnah Shia Sunni debate'},
  'id': 'crava6e3t',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'}]
The statement by the Pakistan military chief Asif Munir that Jinnah was a Shia is controversial because it challenges the widely accepted narrative in Pakistan that Jinnah was a Sunni Muslim. The controversy surrounding Jinnah's religious affiliation is rooted in the fact that Jinnah was born into a Shia family but later identified as a Sunni. Some sources claim that Jinnah remained a Shia Muslim till the end of his life, while others argue that he converted to Sunni Islam. The debate over Jinnah's religious

In [91]:
pprint(response.content)

('The statement by the Pakistan military chief Asif Munir that Jinnah was a '
 'Shia is controversial because it challenges the widely accepted narrative in '
 'Pakistan that Jinnah was a Sunni Muslim. The controversy surrounding '
 "Jinnah's religious affiliation is rooted in the fact that Jinnah was born "
 'into a Shia family but later identified as a Sunni. Some sources claim that '
 'Jinnah remained a Shia Muslim till the end of his life, while others argue '
 "that he converted to Sunni Islam. The debate over Jinnah's religious "
 'affiliation is significant in Pakistan because it has implications for the '
 "country's sectarian identity and its relations with Iran, a predominantly "
 'Shia nation. The controversy has been fueled by the rise of Sunni extremist '
 "groups in Pakistan, which have sought to appropriate Jinnah's legacy and "
 'cast him as a Sunni Muslim. However, evidence suggests that Jinnah was a '
 'secular individual who did not prioritize his personal religious 